# SMA Module 1 — Intro and Data Sources

Wikipedia, Yahoo Finance, and FRED. Run cells from top to bottom.


## 0) Imports


In [ ]:
import os
# keep caches on E: — C: fills up fast on this machine
os.environ.setdefault("MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
os.environ.setdefault("TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("TMP", r"E:\IT_SPACES\AI\.cache\tmp")

import numpy as np
import pandas as pd
import requests
import yfinance as yf

yf.set_tz_cache_location(r"E:\IT_SPACES\AI\.cache\py-yfinance")

from io import StringIO
from datetime import date


## Question 1: S&P 500 additions

Which calendar year, starting from 2020, had the most additions to the current S&P 500 list?

Source: [List of S&P 500 companies](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies).


### Fetch the Wikipedia page


In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
# I got 403 until I sent a browser User-Agent
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()
print("HTTP", response.status_code, "bytes", len(response.text))


### Parse tables

`pandas.read_html` pulls every HTML table. The first one is the current constituents list.


In [ ]:
# pandas treated the HTML as a filepath until I wrapped it in StringIO
tables = pd.read_html(StringIO(response.text))
constituents = tables[0].copy()
constituents.columns


### Tickers, names, and addition year


In [ ]:
df = constituents[["Symbol", "Security", "Date added"]].copy()
df["Date added"] = pd.to_datetime(df["Date added"], errors="coerce")
df["added_year"] = df["Date added"].dt.year
df.head()


### Additions by year since 2020

Count additions among current members. Drop 2026 from the max because the year is still running.


In [ ]:
yearly = (
    df.loc[df["added_year"] >= 2020, "added_year"]
    .value_counts()
    .sort_index()
)
yearly_tbl = yearly.rename("n_added").rename_axis("year").reset_index()
print(yearly_tbl.to_string(index=False))

# count finished years only — the 224 twenty-year names aren't this answer
finished = yearly.loc[yearly.index < 2026]
print("finished years (2020–2025), most additions:", int(finished.idxmax()), "n =", int(finished.max()))


### How long have current members been in the index?

As of 2026-09-05, count constituents whose addition date is at least 20 years earlier.


In [ ]:
asof = pd.Timestamp("2026-09-05")
cutoff = asof - pd.DateOffset(years=20)
# missing dates become False here, so they drop out of the count
over_20 = df["Date added"] <= cutoff
long_names = (
    df.loc[over_20, ["Symbol", "Security", "Date added"]]
    .sort_values("Date added")
    .reset_index(drop=True)
)
print("as-of", asof.date(), "cutoff", cutoff.date())
print("n >= 20 years:", len(long_names), "of", len(df))  # extra — not the Q1 form answer
long_names.head()


## Question 2: World indexes YTD (as of 21 August 2026)

Year-to-date close-to-close growth, 2026-01-01 to 2026-08-21. How many of the ten non-US indexes beat `^GSPC`?

No FX conversion.


### Symbols and window


In [ ]:
tickers = {
    "US": "^GSPC",
    "China": "000001.SS",
    "Hong Kong": "^HSI",
    "Australia": "^AXJO",
    "India": "^NSEI",
    "Canada": "^GSPTSE",
    "Germany": "^GDAXI",
    "UK": "^FTSE",
    "Japan": "^N225",
    "Mexico": "^MXX",
    "Brazil": "^BVSP",
}
start_date = "2026-01-01"
end_date = "2026-08-21"
# yfinance end is exclusive, so I used the next day to keep 21 Aug
download_end = "2026-08-22"


### Daily closes


In [ ]:
closes = {}
for country, symbol in tickers.items():
    hist = yf.Ticker(symbol).history(start=start_date, end=download_end, interval="1d")
    close = hist["Close"].copy()
    close.index = pd.to_datetime(close.index.date)
    closes[country] = close
    print(country, symbol, "rows", len(close), "first", close.index.min().date(), "last", close.index.max().date())


### YTD return and count vs the US


In [ ]:
ytd_rows = []
for country, symbol in tickers.items():
    window = closes[country].loc[start_date:end_date].dropna()
    ytd = window.iloc[-1] / window.iloc[0] - 1
    ytd_rows.append(
        {
            "country": country,
            "symbol": symbol,
            "start_day": window.index[0].date(),
            "end_day": window.index[-1].date(),
            "ytd": ytd,
            "ytd_pct": ytd * 100,
        }
    )
ytd_table = pd.DataFrame(ytd_rows).sort_values("ytd", ascending=False).reset_index(drop=True)
us_ytd = float(ytd_table.loc[ytd_table["country"] == "US", "ytd"].iloc[0])
others = ytd_table.loc[ytd_table["country"] != "US"]
n_better = int((others["ytd"] > us_ytd).sum())

print(ytd_table.to_string(index=False))
print("US YTD %:", round(us_ytd * 100, 2))
print("non-US indexes with higher YTD:", n_better)
print(list(others.loc[others["ytd"] > us_ytd, "country"]))


### Same comparison over 3, 5, and 10 years

Same end date, local-currency closes.


In [ ]:
long_start = (pd.Timestamp(end_date) - pd.DateOffset(years=10) - pd.DateOffset(days=14)).strftime("%Y-%m-%d")
long_closes = {}
for country, symbol in tickers.items():
    hist = yf.Ticker(symbol).history(start=long_start, end=download_end, interval="1d")
    close = hist["Close"].copy()
    close.index = pd.to_datetime(close.index.date)
    long_closes[country] = close

period_rows = []
for years in (3, 5, 10):  # extra 3/5/10y windows — not the form
    pstart = (pd.Timestamp(end_date) - pd.DateOffset(years=years)).strftime("%Y-%m-%d")
    us_window = long_closes["US"].loc[pstart:end_date].dropna()
    us_ret = us_window.iloc[-1] / us_window.iloc[0] - 1
    n_better_long = 0
    for country in tickers:
        if country == "US":
            continue
        window = long_closes[country].loc[pstart:end_date].dropna()
        if len(window) < 2:
            continue
        if window.iloc[-1] / window.iloc[0] - 1 > us_ret:
            n_better_long += 1
    period_rows.append(
        {
            "years": years,
            "start": us_window.index[0].date(),
            "end": us_window.index[-1].date(),
            "us_return_pct": float(us_ret) * 100,
            "n_better_than_us": n_better_long,
        }
    )
pd.DataFrame(period_rows)


## Question 3: S&P 500 corrections

A correction is a drop of at least 5% from the most recent all-time-high close.

Between consecutive all-time highs I take the trough, then
drawdown % = (peak − trough) / peak × 100
and duration = calendar days from peak to trough.


### Daily S&P 500 since 1950


In [ ]:
spx_hist = yf.Ticker("^GSPC").history(start="1950-01-01", interval="1d")
spx_close = spx_hist["Close"].copy()
spx_close.index = pd.to_datetime(spx_close.index.date)
print(len(spx_close), spx_close.index.min().date(), "→", spx_close.index.max().date())
spx_close.tail()


### All-time highs, drawdowns, percentiles


In [ ]:
prev_peak = spx_close.shift(1).cummax()
is_ath = spx_close > prev_peak  # ATH if close beats the prior running max
is_ath.iloc[0] = True
ath_dates = spx_close.index[is_ath]

correction_rows = []
for i in range(len(ath_dates) - 1):
    peak_day = ath_dates[i]
    next_ath = ath_dates[i + 1]
    between = spx_close.loc[peak_day:next_ath].iloc[1:-1]  # trough between those ATHs
    if between.empty:
        continue
    trough_day = between.idxmin()
    peak_px = float(spx_close.loc[peak_day])
    trough_px = float(between.loc[trough_day])
    correction_rows.append(
        {
            "peak_day": peak_day.date(),
            "trough_day": trough_day.date(),
            "drawdown_pct": (peak_px - trough_px) / peak_px * 100,
            "duration_days": (trough_day - peak_day).days,
        }
    )

corrections = pd.DataFrame(correction_rows)
# I kept drawdowns >= 5%; median landed near 8
corr5 = corrections.loc[corrections["drawdown_pct"] >= 5].sort_values(
    "drawdown_pct", ascending=False
).reset_index(drop=True)

print("corrections >= 5%:", len(corr5))
print(corr5.head(10).to_string(index=False))
print("drawdown % 25/50/75:", [round(float(x), 2) for x in corr5["drawdown_pct"].quantile([0.25, 0.5, 0.75])])
print("duration days 25/50/75:", [round(float(x), 2) for x in corr5["duration_days"].quantile([0.25, 0.5, 0.75])])
print("median drawdown %:", round(float(corr5["drawdown_pct"].median()), 1))


## Question 4: Amazon earnings surprises

Two-day return around an announcement day (Day 2):
`Close_Day3 / Close_Day1 − 1`.
Keep `Surprise(%) > 0` and take the median of that 2-day return (in percent).


In [ ]:
amzn = yf.Ticker("AMZN")
earnings = amzn.get_earnings_dates()
print(len(earnings), "rows")
earnings


In [ ]:
amzn_close = amzn.history(period="max", interval="1d")["Close"].copy()
amzn_close.index = pd.to_datetime(amzn_close.index.date)
# Day 2 = announcement; Close3 / Close1 - 1
ret_2d = amzn_close.shift(-1) / amzn_close.shift(1) - 1

earn = earnings.copy()
earn.index = pd.to_datetime(earn.index.date)
surprise_col = "Surprise(%)"
earn["ret_2d"] = ret_2d.reindex(earn.index)
earn["ret_2d_pct"] = earn["ret_2d"] * 100
# one future date has no EPS yet
pos = earn.loc[earn[surprise_col] > 0].dropna(subset=["ret_2d", surprise_col])  # positive surprises only

print("positive surprises:", len(pos))
print("median 2-day %:", round(float(pos["ret_2d_pct"].median()), 2))
print(pos[[surprise_col, "ret_2d"]].corr())
pos[[surprise_col, "ret_2d", "ret_2d_pct"]]


### Surprise vs return, bull vs bear

Split announcement days by whether `^GSPC` is above its 200-day average.


In [ ]:
regime_close = yf.Ticker("^GSPC").history(start="2019-01-01", interval="1d")["Close"].copy()
regime_close.index = pd.to_datetime(regime_close.index.date)
is_bull = regime_close > regime_close.rolling(200).mean()

pos_reg = pos.copy()
# extra bull/bear split — not the form
pos_reg["bull"] = is_bull.reindex(pos_reg.index)
bull_ret = pos_reg.loc[pos_reg["bull"] == True, "ret_2d_pct"]
bear_ret = pos_reg.loc[pos_reg["bull"] == False, "ret_2d_pct"]
print("bull n, median %:", int(bull_ret.count()), round(float(bull_ret.median()), 2) if len(bull_ret) else None)
print("bear n, median %:", int(bear_ret.count()), round(float(bear_ret.median()), 2) if len(bear_ret) else None)
pos_reg[[surprise_col, "ret_2d_pct", "bull"]]


## Question 5: Capstone idea

I want a short-term prediction model for the US stock market, focusing on the S&P 500 and a few large names like AMZN, over about a 30-day horizon after a dip.

Q3 was about 5% drops from all-time highs, and the homework mentioned “buy the dip”, so I would start there. I am not sure about the model yet. For inputs I would reuse the correction size/length from Q3, earnings surprises like Q4, and the FRED series we already pulled in the lesson notebook (Fed funds and core CPI).


## Question 6: Extra metrics for that idea

I reused a few FRED downloads from the Module 1 notebook.


In [ ]:
import pandas_datareader as pdr

start = date(1990, 1, 1)
fedfunds = pdr.DataReader("FEDFUNDS", "fred", start=start)
cpilfesl = pdr.DataReader("CPILFESL", "fred", start=start)
dgs5 = pdr.DataReader("DGS5", "fred", start=start)

print(fedfunds.tail(3))
print(cpilfesl.tail(3))
print(dgs5.tail(3))


- FEDFUNDS — Fed funds rate from class. I want to see if buying the dip looks different when rates are high.
- CPILFESL — core CPI from class. Inflation might change how the market comes back.
- DGS5 — 5-year Treasury from class (we also downloaded DGS1). Extra rates series.

They are not all daily, so for now I would just take the last value before the dip.
